<a href="https://colab.research.google.com/github/Ramkanc/agentic/blob/main/Agent_Manufacturing1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q gradio chromadb sentence-transformers pandas plotly huggingface_hub

In [4]:
csv_file_path = 'data.csv' # Adjust this path to your CSV file

# Read the CSV file into a pandas DataFrame
try:
    df_from_csv = pd.read_csv(csv_file_path)
    print(f"Successfully loaded {len(df_from_csv)} rows from {csv_file_path}")
    display(df_from_csv.head())
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please ensure it's in the correct directory.")
    df_from_csv = pd.DataFrame() # Create an empty DataFrame to prevent errors later

# Convert the DataFrame to a list of dictionaries, matching the initial_data structure
# Assuming the CSV has columns: 'id', 'problem', 'solution', 'model', 'module'
if not df_from_csv.empty:
    try:
        csv_data_list = df_from_csv[['id', 'problem', 'solution', 'model', 'module']].to_dict(orient='records')
        print("\nConverted data to list of dictionaries:")
        print(csv_data_list[:3]) # Display first 3 items for verification
    except KeyError as e:
        print(f"Error: Missing expected column in CSV: {e}. Please check your CSV file's headers.")
        csv_data_list = []
else:
    csv_data_list = []


Successfully loaded 2000 rows from data.csv


,id,problem,solution,model,module
0,ISS-104,Hydraulic fluid leak near the landing gear strut.,Replace the high-pressure seal and check for p...,A220,Landing Gear
1,ISS-105,Brake parachute deployment not arming.,Inspect parachute release pin and replace depl...,B747,Brakes
2,ISS-106,Pitch trim indicator disagrees between captain...,Replace pitch trim transducer and verify signa...,B757,Flight Controls
3,ISS-107,Electric pump motor overheating on ground ops.,Check motor cooling airflow and inspect brushe...,A330,Hydraulics
4,ISS-108,Hydraulic filter differential pressure indicat...,Replace hydraulic filter element and inspect f...,A320,Hydraulics



Converted data to list of dictionaries:
[{'id': 'ISS-104', 'problem': 'Hydraulic fluid leak near the landing gear strut.', 'solution': 'Replace the high-pressure seal and check for piston scoring.', 'model': 'A220', 'module': 'Landing Gear'}, {'id': 'ISS-105', 'problem': 'Brake parachute deployment not arming.', 'solution': 'Inspect parachute release pin and replace deployment initiator.', 'model': 'B747', 'module': 'Brakes'}, {'id': 'ISS-106', 'problem': 'Pitch trim indicator disagrees between captain and FO.', 'solution': 'Replace pitch trim transducer and verify signal output.', 'model': 'B757', 'module': 'Flight Controls'}]


In [5]:
import gradio as gr
import pandas as pd
import plotly.express as px
import chromadb
from sentence_transformers import SentenceTransformer
import uuid

# --- INITIALIZE DATABASE & MODELS ---
# Using a local vector store for manufacturing data
client = chromadb.Client()
collection = client.get_or_create_collection(name="mfg_history")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample Data to pre-populate the system
#
#initial1_data = [
#    {"id": "ISS-101", "problem": "Hydraulic fluid leak near the landing gear strut.", "solution": "Replace the high-pressure seal and check for piston scoring.", "model": "A320", "module": "Landing Gear"},
#    {"id": "ISS-102", "problem": "Fastener torque inconsistency in the wing box assembly.", "solution": "Recalibrate pneumatic drivers and implement a secondary torque check.", "model": "B737", "module": "Wings"},
#    {"id": "ISS-103", "problem": "Avionics bay cooling fan failure during pre-flight.", "solution": "Clear debris from intake and replace the brushless motor unit.", "model": "A350", "module": "Avionics"}
#]
#
initial_data = csv_data_list

# Add initial data to Vector DB
for item in initial_data:
    embedding = embed_model.encode(item["problem"]).tolist()
    collection.add(
        ids=[item["id"]],
        embeddings=[embedding],
        metadatas=[{"solution": item["solution"], "model": item["model"], "module": item["module"]}],
        documents=[item["problem"]]
    )

# --- AGENT FUNCTIONS ---

def retrieve_and_classify(problem_desc, aircraft_model):
    """Retrieves similar issues and 'classifies' the module."""
    # 1. Search Vector DB
    query_vec = embed_model.encode(problem_desc).tolist()
    results = collection.query(query_embeddings=[query_vec], n_results=1)

    # 2. Extract Data
    if results['ids'][0]:
        match_id = results['ids'][0][0]
        match_sol = results['metadatas'][0][0]['solution']
        match_mod = results['metadatas'][0][0]['module']
        return f"✅ **Matched with {match_id}**\n\n**Suggested Solution:** {match_sol}", match_mod
    return "❌ No similar issues found. Please propose a new solution.", "General"

def generate_lessons_learned(target_model):
    """Simulates agentic summarization of lessons learned."""
    # Filter metadata for the model
    all_data = collection.get(include=['metadatas', 'documents'])
    lessons = []
    for meta, doc in zip(all_data['metadatas'], all_data['documents']):
        if meta['model'] == target_model:
            lessons.append(f"• **Problem:** {doc}\n  **Outcome:** {meta['solution']}")

    if not lessons:
        return "No historical data found for this aircraft model."

    summary_header = f"### Lessons Learned Report: {target_model}\n"
    return summary_header + "\n".join(lessons)

def update_dashboard():
    """Generates analytics for the dashboard tab."""
    all_data = collection.get(include=['metadatas'])
    df = pd.DataFrame([m for m in all_data['metadatas']])
    if df.empty: return None, None

    fig1 = px.pie(df, names='model', title="Issues by Aircraft Model", color_discrete_sequence=px.colors.sequential.RdBu)
    fig2 = px.bar(df, x='module', color='model', title="Problem Distribution by Module")
    return fig1, fig2

# --- GRADIO UI LAYOUT ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ✈️ Aerospace Manufacturing: Agentic AI Intelligence")

    with gr.Tabs():
        # TAB 1: NEW ISSUE ENTRY
        with gr.TabItem("🚀 Issue Analysis & Retrieval"):
            with gr.Row():
                with gr.Column():
                    prob_input = gr.TextArea(label="Problem Description", placeholder="Describe the issue here...")
                    with gr.Row():
                        model_input = gr.Dropdown(["A320", "B737", "A350", "B787"], label="Aircraft Model")
                        module_input = gr.Textbox(label="Module Name (Inferred)", interactive=False)
                    btn_analyze = gr.Button("Analyze Issue", variant="primary")

                with gr.Column():
                    output_solution = gr.Markdown(label="Agent Recommendations")

            btn_analyze.click(
                fn=retrieve_and_classify,
                inputs=[prob_input, model_input],
                outputs=[output_solution, module_input]
            )

        # TAB 2: DASHBOARDS
        with gr.TabItem("📊 Analytics Dashboard"):
            btn_refresh = gr.Button("Refresh Dashboard Data")
            with gr.Row():
                plot1 = gr.Plot()
                plot2 = gr.Plot()

            btn_refresh.click(fn=update_dashboard, outputs=[plot1, plot2])

        # TAB 3: LESSONS LEARNED
        with gr.TabItem("🧠 Lessons Learned"):
            model_select = gr.Dropdown(["A320", "B737", "A350", "B787"], label="Select Model for Summary")
            btn_summary = gr.Button("Generate Model Insights")
            lessons_output = gr.Markdown()

            btn_summary.click(fn=generate_lessons_learned, inputs=model_select, outputs=lessons_output)

# Launch the app
demo.launch(debug=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3110/4232249268.py:76: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a4361b7de72b94516b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a4361b7de72b94516b.gradio.live


### Load data from CSV and convert to the required format